In [7]:
import torch
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = 204
model = timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=num_classes).to(device)
checkpoint = torch.load("D:\\Tesi\\FirstFineTuning\\best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [8]:
from torchvision.datasets import ImageFolder
from torchvision import transforms

data_config = timm.data.resolve_model_data_config(model)
imagenet_mean, imagenet_std = data_config["mean"], data_config["std"]

search_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(mean=imagenet_mean, std=imagenet_std)])

path = "D:\\Tesi\\Sets\\Set1\\search"
batch_size = 256

search_set = ImageFolder(root=path, transform=search_transform)
search_loader = torch.utils.data.DataLoader(search_set, batch_size=batch_size, shuffle=False, num_workers=1)

In [9]:
from NAS.NAS_Utils import set_initial_masks, count_parameters, compute_obj, find_target_emb, compute_grads
from torch.nn import CrossEntropyLoss

set_initial_masks(model)
compute_grads(model, CrossEntropyLoss(), device, search_loader)
#compute_obj(model, CrossEntropyLoss(), device, search_loader)

(1.1365426555275917, 0.6708333333333333)

from NAS.NAS_Utils import find_target_emb

print(find_target_emb(model))

from PruneUtils import importance_score

weights = [
    model.patch_embed.proj.weight[2],
    model.patch_embed.proj.bias[2],
    model.patch_embed.proj.weight[3],
    model.patch_embed.proj.bias[3],
]

grads = [
    model.patch_embed.proj.weight.grad[2],
    model.patch_embed.proj.bias.grad[2],
    model.patch_embed.proj.weight.grad[3],
    model.patch_embed.proj.bias.grad[3]
]
print(importance_score(weights, grads))

from PruneUtils import head_alignment, HeadAligned, WeightBias

head_aligned = head_alignment(model.blocks[0].attn)
print(head_aligned.proj.weight.shape)


from NAS.NAS_Utils import find_target_mlp, apply_mlp_Prune

apply_mlp_Prune(model, find_target_mlp(model))
print(count_parameters(model))
print(compute_obj(model, CrossEntropyLoss(), device, search_loader))